In [1]:
import pandas as pd
from multicall import Call
import numpy as np
from web3 import Web3


from mainnet_launch.database.schema.full import (
    RebalancePlan,
    DestinationTokenValues,
    TokenValues,
    Autopools,
    DestinationStates,
    DestinationTokens,
    Destinations,
    AutopoolDestinationStates,
    Tokens,
    AutopoolStates,
    DexSwapSteps,
)
import plotly.express as px
import time
import concurrent.futures
import boto3
from botocore import UNSIGNED
from botocore.config import Config
import json


from mainnet_launch.abis import STATS_CALCULATOR_REGISTRY_ABI
from mainnet_launch.data_fetching.get_events import fetch_events


from mainnet_launch.database.schema.postgres_operations import (
    get_full_table_as_orm,
    get_full_table_as_df,
    insert_avoid_conflicts,
    get_subset_not_already_in_column,
    natural_left_right_using_where,
)
from mainnet_launch.data_fetching.get_state_by_block import (
    get_raw_state_by_blocks,
    safe_normalize_with_bool_success,
    build_blocks_to_use,
    identity_with_bool_success,
    get_state_by_one_block,
)
from mainnet_launch.constants import ALL_CHAINS, ALL_AUTOPOOLS, ROOT_PRICE_ORACLE, ChainData, AutopoolConstants

from mainnet_launch.pages.autopool_diagnostics.lens_contract import (
    fetch_pools_and_destinations_df,
)

In [ ]:
def convert_rebalance_plan_json_to_rebalance_plan_line(
    rebalance_plan_json_key: str, s3_client, autopool: AutopoolConstants
):
    plan = json.loads(
        s3_client.get_object(
            Bucket=autopool.solver_rebalance_plans_bucket,
            Key=rebalance_plan_json_key,
        )["Body"].read()
    )

    plan["rebalance_plan_json_key"] = rebalance_plan_json_key
    plan["autopool_vault_address"] = autopool.autopool_eth_addr
    return plan


def _extract_rebalance_plan_and_dex_steps(
    plan: dict, tokens: list[Tokens], destinations: list[Destinations]
) -> tuple[RebalancePlan, list[DexSwapSteps]]:

    dest_map = {Web3.toChecksumAddress(d.destination_vault_address): d.underlying_symbol for d in destinations}
    token_map = {Web3.toChecksumAddress(t.token_address): t.decimals for t in tokens}

    # then pull out your four values in two lines
    out_addr = Web3.toChecksumAddress(plan["destinationOut"])
    in_addr = Web3.toChecksumAddress(plan["destinationIn"])
    underlying_out_symbol = dest_map[out_addr]
    underlying_in_symbol = dest_map[in_addr]

    out_token = Web3.toChecksumAddress(plan["tokenOut"])
    in_token = Web3.toChecksumAddress(plan["tokenIn"])
    token_out_decimals = token_map[out_token]
    token_in_decimals = token_map[in_token]
    in_destination_name = plan["rebalanceTest"]["inDest"]

    for i, d in zip(plan["addRank"]):
        if d[0] == in_destination_name:
            # not sure here on decimals
            candidate_destinations_rank = i
            projected_net_gain = d[1] / 1e18
            projected_swap_cost = d[2] / 1e18
            projected_gross_gain = projected_swap_cost + projected_net_gain

    new_rebalance_plan_row = RebalancePlan(
        file_name=plan["rebalance_plan_json_key"],
        datetime_generated=pd.to_datetime(int(plan["timestamp"]), unit="s", utc=True),
        autopool_vault_address=plan["autopool_vault_address"],
        chain_id=int(plan["chainId"]),
        dex_aggregator=plan["steps"][0]["dex"],  # not certain here
        solver_address=Web3.toChecksumAddress(plan["solverAddress"]),
        rebalance_type=plan["rebalanceTest"]["type"],
        destination_in=Web3.toChecksumAddress(plan["destinationOut"]),
        token_in=Web3.toChecksumAddress(plan["tokenOut"]),
        destination_out=Web3.toChecksumAddress(plan["destinationIn"]),
        token_out=Web3.toChecksumAddress(plan["tokenIn"]),
        move_name=f"{underlying_out_symbol} -> {underlying_in_symbol}",
        # NOTE: this might amountOutETH might be different for autoUSD, not certain what decimals it is
        amount_out=int(plan["amountOut"]) / (10**token_out_decimals),
        amount_out_safe_value=int(plan["amountOutETH"]) / (10**token_out_decimals),
        min_amount_in=int(plan["minAmountIn"]) / (10**token_in_decimals),
        min_amount_in_safe_value=int(plan["minAmountInETH"]) / (10**token_in_decimals),
        # rebalanceTest values
        out_spot_eth=int(plan["rebalanceTest"]["outSpotETH"]) / (10**token_out_decimals),
        out_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
        in_spot_eth=int(plan["rebalanceTest"]["inSpotETH"]) / (10**token_in_decimals),
        in_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
        in_dest_adj_apr=float(plan["rebalanceTest"]["inDestAdjApr"]),
        apr_delta=float(plan["rebalanceTest"]["inDestAdjApr"]) - float(plan["rebalanceTest"]["outDestApr"]),
        swap_offset_period=int(plan["rebalanceTest"]["swapOffsetPeriod"]),
        num_candidate_destinations=len(plan["addRank"]),
        candidate_destinations_rank=candidate_destinations_rank,
        projected_swap_cost=projected_swap_cost,
        projected_net_gain=projected_net_gain,
        projected_gross_gain=projected_gross_gain,
        projected_slippage=100
        * projected_swap_cost
        / int(plan["rebalanceTest"]["outSpotETH"])
        / (10**token_out_decimals),  # out spot eth
    )

    new_dex_steps = []

    for step_index, step in enumerate(plan["steps"]):
        if step["type"] == "swap":
            new_dex_steps.append(
                DexSwapSteps(file_name=plan["rebalance_plan_json_key"], step_index=step_index, dex=step["dex"])
            )

    return new_rebalance_plan_row, new_dex_steps


def ensure_rebalance_plan_table_is_current():
    s3_client = boto3.client("s3", config=Config(signature_version=UNSIGNED))

    for autopool in ALL_AUTOPOOLS:

        solver_plan_paths_on_remote = [
            r["Key"] for r in s3_client.list_objects_v2(Bucket=autopool.solver_rebalance_plans_bucket).get("Contents")
        ]
        plans_not_already_fetched = get_subset_not_already_in_column(
            RebalancePlan,
            RebalancePlan.file_name,
            solver_plan_paths_on_remote,
            where_clause=RebalancePlan.autopool_vault_address == autopool.autopool_eth_addr,
        )

        tokens = get_full_table_as_orm(Tokens, where_clause=Tokens.chain_id == autopool.chain.chain_id)
        destinations = get_full_table_as_orm(
            Destinations, where_clause=Destinations.chain_id == autopool.chain.chain_id
        )

        all_rebalance_plan_rows = []
        all_dex_steps_rows = []
        for plan_on_remote in plans_not_already_fetched:
            plan = convert_rebalance_plan_json_to_rebalance_plan_line(plan_on_remote, s3_client, autopool)
            new_rebalance_plan_row, new_dex_steps_rows = _extract_rebalance_plan_and_dex_steps(
                plan, tokens, destinations
            )
            all_rebalance_plan_rows.append(new_rebalance_plan_row)
            all_dex_steps_rows.extend(new_dex_steps_rows)

    insert_avoid_conflicts(all_rebalance_plan_rows, RebalancePlan, index_elements=[RebalancePlan.file_name])

    insert_avoid_conflicts(
        all_dex_steps_rows, DexSwapSteps, index_elements=[DexSwapSteps.file_name, DexSwapSteps.step_index]
    )


plan, destinations, tokens = ensure_rebalance_plan_table_is_current()
plan

2025-04-29 11:52:25,480 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-29 11:52:25,481 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM rebalance_plan
            WHERE rebalance_plan.autopool_vault_address = '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56'
        
2025-04-29 11:52:25,481 INFO sqlalchemy.engine.Engine [cached since 75.68s ago] {}
2025-04-29 11:52:25,664 INFO sqlalchemy.engine.Engine COMMIT
2025-04-29 11:52:25,754 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-29 11:52:25,755 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM tokens
            WHERE tokens.chain_id = 1
        
2025-04-29 11:52:25,755 INFO sqlalchemy.engine.Engine [cached since 75.67s ago] {}
2025-04-29 11:52:25,950 INFO sqlalchemy.engine.Engine COMMIT
2025-04-29 11:52:26,043 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-04-29 11:52:26,044 INFO sqlalchemy.engine.Engine 
            SELECT *
            FROM destinations
            WHERE destinat

KeyError: '0xC8Eb2Cf2f792F77AF0Cd9e203305a585E588179D'

In [ ]:
# def _extract_rebalance_plan_and_dex_steps(
#     plan: dict, tokens: list[Tokens], destinations: list[Destinations]
# ) -> tuple[RebalancePlan, list[DexSwapSteps]]:

#     underlying_out_symbol = [
#         d.underlying_symbol
#         for d in destinations
#         if d.destination_vault_address == Web3.toChecksumAddress(plan["destinationOut"])
#     ][0]

#     underlying_in_symbol = [
#         d.underlying_symbol
#         for d in destinations
#         if d.destination_vault_address == Web3.toChecksumAddress(plan["destinationIn"])
#     ][0]

#     token_out_decimals = [t.decimals for t in tokens if t.token_address == Web3.toChecksumAddress(plan["tokenOut"])][0]

#     token_in_decimals = [t.decimals for t in tokens if t.token_address == Web3.toChecksumAddress(plan["tokenIn"])][0]

#     in_destination_name = plan["rebalanceTest"]["inDest"]

#     for i, d in zip(plan["addRank"]):
#         if d[0] == in_destination_name:
#             # not sure here on decimals
#             candidate_destinations_rank = i
#             projected_net_gain = d[1] / 1e18
#             projected_swap_cost = d[2] / 1e18
#             projected_gross_gain = projected_swap_cost + projected_net_gain

#     new_rebalance_plan_row = RebalancePlan(
#         file_name=plan["rebalance_plan_json_key"],
#         datetime_generated=pd.to_datetime(int(plan["timestamp"]), unit="s", utc=True),
#         autopool_vault_address=plan["autopool_vault_address"],
#         chain_id=int(plan["chainId"]),
#         dex_aggregator=plan["steps"][0]["dex"],  # not certain here
#         solver_address=Web3.toChecksumAddress(plan["solverAddress"]),
#         rebalance_type=plan["rebalanceTest"]["type"],
#         destination_in=Web3.toChecksumAddress(plan["destinationOut"]),
#         token_in=Web3.toChecksumAddress(plan["tokenOut"]),
#         destination_out=Web3.toChecksumAddress(plan["destinationIn"]),
#         token_out=Web3.toChecksumAddress(plan["tokenIn"]),
#         move_name=f"{underlying_out_symbol} -> {underlying_in_symbol}",
#         # NOTE: this might amountOutETH might be different for autoUSD, not certain what decimals it is
#         amount_out=int(plan["amountOut"]) / (10**token_out_decimals),
#         amount_out_safe_value=int(plan["amountOutETH"]) / (10**token_out_decimals),
#         min_amount_in=int(plan["minAmountIn"]) / (10**token_in_decimals),
#         min_amount_in_safe_value=int(plan["minAmountInETH"]) / (10**token_in_decimals),
#         # rebalanceTest values
#         out_spot_eth=int(plan["rebalanceTest"]["outSpotETH"]) / (10**token_out_decimals),
#         out_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
#         in_spot_eth=int(plan["rebalanceTest"]["inSpotETH"]) / (10**token_in_decimals),
#         in_dest_apr=float(plan["rebalanceTest"]["outDestApr"]),
#         in_dest_adj_apr=float(plan["rebalanceTest"]["inDestAdjApr"]),
#         apr_delta=float(plan["rebalanceTest"]["inDestAdjApr"]) - float(plan["rebalanceTest"]["outDestApr"]),
#         swap_offset_period=int(plan["rebalanceTest"]["swapOffsetPeriod"]),
#         num_candidate_destinations=len(plan["addRank"]),
#         candidate_destinations_rank=candidate_destinations_rank,
#         projected_swap_cost=projected_swap_cost,
#         projected_net_gain=projected_net_gain,
#         projected_gross_gain=projected_gross_gain,
#         projected_slippage=100
#         * projected_swap_cost
#         / int(plan["rebalanceTest"]["outSpotETH"])
#         / (10**token_out_decimals),  # out spot eth
#     )

#     new_dex_steps = []

#     for i, step in enumerate(plan["steps"]):
#         if step["type"] == "swap":
#             new_dex_steps.append(DexSwapSteps(file_name=plan["rebalance_plan_json_key"], step_index=i, dex=step["dex"]))

#     return new_rebalance_plan_row, new_dex_steps

IndentationError: expected an indented block after 'for' statement on line 23 (3826577108.py, line 24)

In [ ]:
# the first one is net gain (gain - cost) over swap cost offset period
# in base asset units (decimals 18). 2nd one is swap cost (same units as first).


#          'rebalanceTest': {'currentTimestamp': 1745847039,
#   'type': 'FromIdle',
#   'outDest': 'Autopool',
#   'outSpotETH': 1.2886584553580578e+20,
#   'outDestApr': 0,
#   'inDest': 'Tokemak-Wrapped Ether-osETH/rETH',
#   'inSpotETH': 1.2882292987566273e+20,
#   'inDestApr': 0.06753612298754073,
#   'inDestAdjApr': 0.06439188622228886,
#   'swapOffsetPeriod': 40},


# 'addRank': [['Tokemak-Wrapped Ether-osETH/rETH',
#    8.017298132176219e+17, # (net gain during swap ost offset period), swap cost
#    1.073271250808832e+17], # swap co
#   ['Tokemak-Wrapped Ether-Balancer weETH/rETH StablePool',
#    6.846795268620925e+17,
#    1.682938657221673e+17],
#   ['Tokemak-Wrapped Ether-pxETH/wETH',
#    6.090233780379204e+17,
#    9.69917050570834e+16],
#   ['Tokemak-Wrapped Ether-pxETH/stETH',
#    6.038320873790577e+17,
#    1.2142891437185434e+17],
#   ['Tokemak-Wrapped Ether-Balancer pxETH/wETH StablePool',
#    5.860874308686176e+17,
#    9.470690111361843e+16]],
#     )

In [ ]:
plan

{'timestamp': 1745847039,
 'sodOnly': False,
 'chainId': '1',
 'solverAddress': '0x952D7a7eB2e0804d37d9244BE8e47341356d2f5D',
 'poolAddress': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'destinationOut': '0x0A2b94F6871c1D7A32Fe58E1ab5e6deA2f114E56',
 'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
 'amountOut': '128865845535805784064',
 'amountOutETH': '128865845535805784064',
 'destinationIn': '0x3F55eedDe51504E6Ed0ec30E8289b4Da11EdB7F9',
 'tokenIn': '0xe080027Bd47353b5D1639772b4a75E9Ed3658A0d',
 'minAmountIn': '123375446482083250176',
 'minAmountInETH': '128820552879264989184',
 'steps': [{'stepType': 'swap',
   'dex': '0xV2',
   'tokenOut': '0xC02aaA39b223FE8D0A0e5C4F27eAD9083C756Cc2',
   'amountOut': '61304214261994627072',
   'tokenIn': '0xf1C9acDc66974dFB6dEcB12aA385b9cD01190E38',
   'minAmountIn': '0',
   'payload': {'blockNumber': '22367874',
    'buyAmount': '58615327036342083166',
    'buyToken': '0xf1c9acdc66974dfb6decb12aa385b9cd01190e38',
    'fees': {'integra

In [ ]:
# # needed
# class RebalancePlan(Base):
#     __tablename__ = "rebalance_plan"

#     file_name: Mapped[str] = mapped_column(nullable=False, primary_key=True)

#     datetime_generated: Mapped[pd.Timestamp] = mapped_column(DateTime(timezone=True), nullable=False)
#     autopool_vault_address: Mapped[str] = mapped_column(nullable=False)
#     chain_id: Mapped[int] = mapped_column(nullable=False)

#     dex_aggregator: Mapped[str] = mapped_column(nullable=False)

#     solver_address: Mapped[str] = mapped_column(nullable=False)
#     rebalance_type: Mapped[str] = mapped_column(nullable=False)

#     # sometimes this has different destinations but the same underlying token. that means
#     destination_out: Mapped[str] = mapped_column(nullable=False)
#     token_out: Mapped[str] = mapped_column(nullable=False)

#     destination_in: Mapped[str] = mapped_column(nullable=False)
#     token_in: Mapped[str] = mapped_column(nullable=False)

#     move_name: Mapped[str] = mapped_column(nullable=False)  # f"{data['destinationOut']} -> {data['destinationIn']}"

#     amount_out: Mapped[float] = mapped_column(nullable=False)

#     # verify if this is safe or spot values
#     amount_out_safe_value: Mapped[float] = mapped_column(nullable=False)
#     min_amount_in_safe_value: Mapped[float] = mapped_column(nullable=False)
#     min_amount_in: Mapped[float] = mapped_column(nullable=False)

#     out_spot_eth: Mapped[float] = mapped_column(nullable=False)
#     out_dest_apr: Mapped[float] = mapped_column(nullable=False)

#     in_dest_apr: Mapped[float] = mapped_column(nullable=False)
#     int_spot_eth: Mapped[float] = mapped_column(nullable=False)
#     in_dest_adj_apr: Mapped[float] = mapped_column(nullable=False)

#     apr_delta: Mapped[float] = mapped_column(nullable=False)
#     swap_offset_period: Mapped[int] = mapped_column(nullable=False)

#     candidate_destinations: Mapped[list[str]] = mapped_column(ARRAY(String), nullable=False)
#     candidate_destinations_rank: Mapped[int] = mapped_column(nullable=False)

#     projected_swap_cost: Mapped[float] = mapped_column(nullable=False)
#     projected_slippage: Mapped[float] = mapped_column(nullable=False)

#     __table_args__ = (
#         ForeignKeyConstraint(
#             ["destination_in", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#         ForeignKeyConstraint(
#             ["destination_out", "chain_id"],
#             ["destinations.destination_vault_address", "destinations.chain_id"],
#         ),
#         ForeignKeyConstraint(["token_in", "chain_id"], ["tokens.token_address", "tokens.chain_id"]),
#         ForeignKeyConstraint(["token_out", "chain_id"], ["tokens.token_address", "tokens.chain_id"]),
#         ForeignKeyConstraint(
#             ["autopool_vault_address", "chain_id"], ["autopools.autopool_vault_address", "autopools.chain_id"]
#         ),
#     )

#     # dex steps here?